
# 🛡️ Hands-on Project: AI Insurance Policy Extractor

Build an LLM application that reads insurance policy documents and extracts validated facts.

**Learning goals**
- Design nested Pydantic models for insurance data.
- Configure Groq or Ollama using environment variables.
- Write system and user prompts for reliable extraction.
- Extract text from PDF/DOCX/TXT documents.
- Validate JSON with Pydantic and handle failures.
- Preserve evidence quotes and page references.

**Safety rule:** Never invent missing values. Use `None` for unknown scalar fields and empty lists for unknown collections.

> Keep your real `.env` private. Never paste API keys into notebook cells or commit them to Git.



## Step 0 — Install and configure

Run from the project root:

```bash
pip install -r requirements.txt
```

Your `.env` can use the configuration you already have:

```env
LLM_PROVIDER=groq
GROQ_API_KEY=your_key
# Optional:
# LLM_MODEL=openai/gpt-oss-120b
# JUDGE_MODEL=openai/gpt-oss-20b
```

For local Ollama, set `LLM_PROVIDER=ollama` and make sure the configured model is pulled and Ollama is running.

This project uses the OpenAI Python client against the OpenAI-compatible Groq/Ollama endpoints. The `JUDGE_MODEL` variable is reserved for a later evaluation extension.



## Step 1 — Define the Pydantic output schema

We need a nested schema because a policy contains multiple categories and repeated items.

Fields are intentionally optional where the document may not contain the information. Do not make a field required merely because it is useful to have.


In [ ]:

from src.schemas import (
    InsurancePolicyExtraction,
    PersonDetails,
    PremiumDetails,
    CoverageItem,
    WaitingPeriod,
)

print(InsurancePolicyExtraction.model_json_schema())



### Schema design checklist

- `Literal` restricts policy type and payment frequency.
- `Field(ge=0)` prevents negative monetary values.
- Nested models keep policyholder, premium, benefits, and evidence organized.
- `date` validates ISO-compatible dates.
- `Evidence` stores the exact quote and PDF page number supporting an extracted field.

Try adding a new field, such as `grace_period_days`, only after deciding how it should behave when absent.



## Step 2 — Configure the LLM client

The client reads `.env` automatically. No API key should be hardcoded in the notebook.


In [ ]:

from src.llm_client import client, MODEL, PROVIDER

print("Provider:", PROVIDER)
print("Model:", MODEL)
print("Client initialized:", client is not None)



## Step 3 — Understand the prompts

**System prompt:** establishes the extractor's role and strict rules: use only source text, treat document contents as untrusted, do not infer missing values, and return JSON.

**User prompt:** supplies the current document excerpt and the Pydantic JSON schema.

The policy document itself is untrusted input. A sentence inside a PDF that says "ignore previous instructions" must be treated as document text, not as an instruction to the model.


In [ ]:

from src.extractor import SYSTEM_PROMPT

print(SYSTEM_PROMPT)



## Step 4 — Read and chunk a policy document

Put a sample policy in `data/`. Use a document you are authorized to process. The reader supports text-based PDF, DOCX, and TXT.

Scanned PDFs require OCR; this starter version will stop if it cannot extract text.


In [ ]:

from src.document_reader import read_document, chunk_pages

DOCUMENT_PATH = "data/sample_policy.pdf"  # Change this path

pages = read_document(DOCUMENT_PATH)
print("Pages/sections:", len(pages))
print("First text preview:")
print(pages[0]["text"][:1500] if pages else "No text found")

chunks = chunk_pages(pages, max_chars=12000)
print("Chunks:", len(chunks))



## Step 5 — Call the LLM and validate the response

The extraction function:
1. Reads the document.
2. Splits long text into chunks.
3. Calls the selected model with JSON mode.
4. Parses the response with `model_validate_json`.
5. Merges partial results when the document has multiple chunks.

Pydantic validation checks structure and types. It does **not** prove that the extracted facts are true; verify the evidence against the source.


In [ ]:

from src.extractor import extract_policy
from pydantic import ValidationError

try:
    extraction = extract_policy(DOCUMENT_PATH)
    print(extraction.model_dump_json(indent=2))
except ValidationError as exc:
    print("The LLM returned data that did not match the schema:")
    print(exc)
except Exception as exc:
    print(f"Extraction failed: {type(exc).__name__}: {exc}")



## Step 6 — Inspect evidence and save JSON

Always inspect evidence quotes and page references before trusting an extracted value.


In [ ]:

from pathlib import Path
import json

# Run this after Step 5 succeeds.
output_path = Path("outputs/extraction.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(extraction.model_dump_json(indent=2), encoding="utf-8")

for item in extraction.evidence:
    print({
        "field": item.field_name,
        "page": item.page_number,
        "quote": item.quote,
    })

print("Saved:", output_path)



## Step 7 — Test validation failures

Pydantic should reject invalid values, such as a negative premium. This is a local schema test and does not call the LLM.


In [ ]:

from pydantic import ValidationError

try:
    PremiumDetails(amount=-500, currency="INR", payment_frequency="monthly")
except ValidationError as exc:
    print("Expected validation failure:")
    print(exc)



## Step 8 — Try these experiments

1. Test a health insurance policy and a motor insurance policy.
2. Check whether policyholder and insured person are extracted separately.
3. Find an exclusion and verify its exact evidence quote.
4. Test a document with a missing policy number; it should remain `null`.
5. Test a scanned PDF; confirm the application reports that OCR is required.
6. Compare extraction results using Groq and Ollama.
7. Add unit tests with synthetic policy text and expected output.

## Completion criteria

- [ ] Pydantic schema represents the target fields.
- [ ] `.env` configures Groq or Ollama without exposing secrets.
- [ ] Document text is extracted and chunked.
- [ ] LLM output is parsed and validated.
- [ ] Unknown values are not fabricated.
- [ ] Evidence is available for manual review.
- [ ] JSON output is saved successfully.

**Production note:** This is an extraction assistant, not a coverage adjudication system. A human reviewer must verify policy facts before business or customer decisions.
